# Benchmark RAGAS — Assistant RAG multimodal

Ce notebook évalue la qualité du système RAG du projet avec **RAGAS**, en réutilisant
**exactement le même pipeline que l'API** (`/api/chat`) :

1. embedding de la question (`hf_client.embed_texts`),
2. retrieval hybride dense + BM25 (`vector_store.hybrid_query`),
3. génération de la réponse sourcée (`mistral_client.generate_answer`).

Les métriques calculées :

| Métrique | Ce qu'elle mesure | A besoin de |
|---|---|---|
| **Faithfulness** | La réponse est-elle fidèle aux passages récupérés (pas d'hallucination) ? | réponse + contextes |
| **Response relevancy** | La réponse répond-elle vraiment à la question ? | réponse + embeddings |
| **Context precision** | Les passages récupérés sont-ils pertinents (peu de bruit) ? | contextes + référence |
| **Context recall** | Le retrieval a-t-il ramené l'info nécessaire pour répondre ? | contextes + référence |

**Choix d'implantation** : RAGAS a besoin d'un LLM « juge » et d'un modèle d'embeddings.
On réutilise les modèles **déjà utilisés par le projet** (Mistral comme juge, et **Mistral
embed** (`mistral-embed`) comme modèle d'embeddings, via `EMBEDDING_BACKEND=mistral`) plutôt
qu'OpenAI : pas de provider supplémentaire, et le juge parle la même langue technique que le
système évalué. Le projet utilisait auparavant `multilingual-e5-large` via l'Inference API HF,
mais Zscaler bloque cet accès sur le PC de stage — les embeddings ont donc basculé sur Mistral
(cf. `CLAUDE.md`).

## 0. Dépendances

À exécuter une seule fois. Installation en scope utilisateur (pas de droits admin requis,
cf. contraintes PC verrouillé). `ragas` tire `langchain-core`, `datasets`, etc.

In [1]:
# %pip install --user ragas langchain-mistralai pandas

## 1. Setup : chemins, SSL, et chargement du pipeline du backend

On se place dans `backend/` pour que `settings` lise `backend/.env` et que le chemin de la
base Chroma (`./data/chroma`) corresponde exactement à celui de l'API. `truststore` règle les
erreurs SSL dues à l'inspection TLS Zscaler (cf. `CLAUDE.md`).

In [2]:
# %pip install truststore
# %pip install tenacity
# %pip install chromadb
# %pip install rank_bm25
# %pip install mistralai

In [3]:
import os
import sys
from pathlib import Path

# notebooks/ -> ../backend
BACKEND = (Path.cwd() / ".." / "backend").resolve()
if not (BACKEND / "app").exists():
    raise RuntimeError(f"Backend introuvable à {BACKEND} — lancez ce notebook depuis notebooks/")

os.chdir(BACKEND)            # pour que settings.data_dir == backend/data (même Chroma que l'API)
sys.path.insert(0, str(BACKEND))

import truststore
truststore.inject_into_ssl()  # SSL via le magasin Windows (Zscaler)

from app.config import settings
from app.services import hf_client, vector_store, mistral_client

n_chunks = vector_store._collection().count()
embedding_model = (
    settings.MISTRAL_EMBED_MODEL if settings.EMBEDDING_BACKEND == "mistral" else settings.HF_EMBEDDING_MODEL
)
print(f"Collection Chroma : {n_chunks} chunks indexés")
print(f"LLM génération : {settings.MISTRAL_MODEL}")
print(f"Embeddings       : {embedding_model} (backend={settings.EMBEDDING_BACKEND})")
print(f"RAG_TOP_K        : {settings.RAG_TOP_K}")
if n_chunks == 0:
    print("\n⚠️  Aucun chunk indexé : ingestez des documents avant de lancer le benchmark.")

c:\Users\aamllal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection Chroma : 178 chunks indexés
LLM génération : mistral-small-latest
Embeddings       : mistral-embed (backend=mistral)
RAG_TOP_K        : 5


## 2. Jeu d'évaluation

Chaque entrée = une `question` + une `reference` (réponse de référence écrite à la main).
La référence sert à `context_precision` et `context_recall`.

> Les questions ci-dessous ciblent les documents **actuellement indexés** (BERT, schémas
> électriques). **À adapter / étoffer** à votre corpus : un benchmark crédible vise ~20-50
> questions variées (factuelles, multi-passages, et quelques-unes *sans réponse* dans le corpus
> pour tester l'aveu d'ignorance).

In [4]:
eval_set = [
    {
        "question": "Que signifie l'acronyme BERT ?",
        "reference": "BERT signifie Bidirectional Encoder Representations from Transformers.",
    },
    {
        "question": "Sur quelles deux tâches non supervisées BERT est-il pré-entraîné ?",
        "reference": (
            "BERT est pré-entraîné sur le Masked Language Model (MLM), où des tokens aléatoires "
            "sont masqués puis prédits, et sur la Next Sentence Prediction (NSP)."
        ),
    },
    {
        "question": "Quel pourcentage de tokens est masqué lors du pré-entraînement MLM de BERT ?",
        "reference": "15 % des tokens WordPiece de chaque séquence sont masqués aléatoirement.",
    },
    {
        "question": "Quel token spécial est ajouté au début de chaque séquence d'entrée de BERT ?",
        "reference": (
            "Le token spécial [CLS] est ajouté au début de chaque séquence ; son état caché "
            "final sert de représentation agrégée pour les tâches de classification."
        ),
    },
    {
        "question": "À quoi sert un schéma de circuit électrique (circuit diagram) ?",
        "reference": (
            "Un schéma de circuit représente, à l'aide de symboles normalisés, les composants "
            "d'un circuit électrique et les connexions électriques entre eux."
        ),
    },
    {
        "question": "En électronique, que représente le symbole de la mise à la terre (ground) ?",
        "reference": (
            "Le symbole de mise à la terre (ground) représente un point de référence commun à "
            "potentiel zéro dans le circuit, souvent un retour commun ou une connexion à la terre."
        ),
    },
]
print(f"{len(eval_set)} questions dans le jeu d'évaluation")

6 questions dans le jeu d'évaluation


## 3. Exécution du pipeline RAG sur chaque question

`run_rag` reproduit le cœur de `post_chat` (sans l'authentification ni la persistance des
conversations). Pour chaque question on récupère la **réponse générée** et la **liste des
passages récupérés** (les `retrieved_contexts` que RAGAS analysera).

In [5]:
def run_rag(question: str, top_k: int | None = None, document_ids: list[str] | None = None):
    """Exécute le pipeline RAG complet et renvoie (réponse, contextes, chunks bruts)."""
    top_k = top_k or settings.RAG_TOP_K
    embedding = hf_client.embed_texts([question])[0]
    chunks = vector_store.hybrid_query(embedding, question, top_k=top_k, document_ids=document_ids)
    answer = mistral_client.generate_answer(question, chunks)
    contexts = [c.text for c in chunks]
    return answer, contexts, chunks

In [6]:
samples = []
for i, item in enumerate(eval_set, 1):
    answer, contexts, chunks = run_rag(item["question"])
    samples.append(
        {
            "user_input": item["question"],
            "retrieved_contexts": contexts,
            "response": answer,
            "reference": item["reference"],
        }
    )
    srcs = ", ".join(sorted({f"{c.document_name} p.{c.page}" for c in chunks}))
    print(f"[{i}/{len(eval_set)}] {item['question']}")
    print(f"    → {answer[:160]}{'…' if len(answer) > 160 else ''}")
    print(f"    sources: {srcs}\n")

[1/6] Que signifie l'acronyme BERT ?
    → L'acronyme **BERT** signifie **"Bidirectional Encoder Representations from Transformers"** [Source 5].
    sources: BERT.pdf p.15, BERT.pdf p.3, BERT.pdf p.None

[2/6] Sur quelles deux tâches non supervisées BERT est-il pré-entraîné ?
    → BERT est pré-entraîné sur deux tâches non supervisées : la **Masked Language Modeling (Mask LM)** et la **Next Sentence Prediction (NSP)**.

Voici la figure ill…
    sources: BERT.pdf p.15, BERT.pdf p.3, BERT.pdf p.None

[3/6] Quel pourcentage de tokens est masqué lors du pré-entraînement MLM de BERT ?
    → D'après les sources fournies, **15 % des tokens** sont masqués lors du pré-entraînement MLM de BERT.

Voici la justification :
- *"Compared to standard langauge…
    sources: BERT.pdf p.None

[4/6] Quel token spécial est ajouté au début de chaque séquence d'entrée de BERT ?
    → Le token spécial ajouté au début de chaque séquence d'entrée de BERT est le token **[CLS]** [Source 2, Source 3].
    sources

## 4. Configuration du juge RAGAS (Mistral + embeddings du projet)

- **LLM juge** : `ChatMistralAI` (même API que la génération). On peut prendre un modèle plus
  fort que celui de production pour juger (ex. `mistral-large-latest`).
- **Embeddings** : on enveloppe `hf_client.embed_texts` dans une classe `langchain` minimale,
  pour que `response_relevancy` utilise **exactement** les embeddings du système — à savoir
  `mistral-embed` (via `EMBEDDING_BACKEND=mistral`), `hf_client.embed_texts` se contentant de
  déléguer à `mistral_client.embed_texts` dans cette configuration.

In [7]:
# %pip install langchain_core
# %pip install langchain_mistralai
# ragas>=0.3.4 (jusqu'à 0.4.x) tire scikit-network, qui doit compiler des extensions C++
# (Cython) — impossible sans droits admin / MSVC Build Tools sur ce PC. 0.3.0-0.3.3 n'a pas
# cette dépendance et couvre toutes les métriques utilisées ici (vérifié sur le métadata PyPI).
# %pip install "ragas<0.3.4"

In [ ]:
import asyncio
import sys
import types

# ragas==0.3.3 importe en dur `langchain_community.chat_models.vertexai`, supprimé de
# langchain_community depuis la 0.4 (VertexAI a été déplacé vers le package séparé
# langchain-google-vertexai). On n'utilise jamais VertexAI ici (seulement ChatMistralAI),
# donc on fournit un stub en mémoire plutôt que de downgrader toute la stack langchain
# (ce qui casserait langchain_mistralai, installé en v1.x).
if "langchain_community.chat_models.vertexai" not in sys.modules:
    _vertexai_stub = types.ModuleType("langchain_community.chat_models.vertexai")

    class ChatVertexAI:  # placeholder non fonctionnel, jamais instancié
        pass

    _vertexai_stub.ChatVertexAI = ChatVertexAI
    sys.modules["langchain_community.chat_models.vertexai"] = _vertexai_stub

# ragas.executor appelle nest_asyncio.apply() sans condition dès son import (nécessaire pour
# tourner dans la boucle asyncio déjà active du kernel Jupyter). Mais nest_asyncio casse le
# suivi de la "current task" dont dépend asyncio.timeout() (utilisé par asyncio.wait_for sur
# Python 3.11+) -> "RuntimeError: Timeout should be used inside a task" dès qu'une métrique
# ragas applique un timeout par échantillon. On remplace asyncio.wait_for par l'ancienne
# implémentation (pré-3.11, basée sur call_later/cancel) qui n'a pas besoin de current_task().
async def _wait_for_without_timeout_cm(fut, timeout):
    if timeout is None:
        return await fut
    fut = asyncio.ensure_future(fut)
    loop = asyncio.get_event_loop()
    handle = loop.call_later(timeout, fut.cancel)
    try:
        return await fut
    except asyncio.CancelledError:
        raise asyncio.TimeoutError from None
    finally:
        handle.cancel()


asyncio.wait_for = _wait_for_without_timeout_cm

In [9]:
from langchain_core.embeddings import Embeddings
from langchain_mistralai import ChatMistralAI

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


class ProjectEmbeddings(Embeddings):
    """Adapte les embeddings HF du projet à l'interface langchain pour RAGAS."""

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return hf_client.embed_texts(list(texts))

    def embed_query(self, text: str) -> list[float]:
        return hf_client.embed_texts([text])[0]


# mistral-large-latest = meilleur juge ; bascule sur le modèle de prod s'il n'est pas dispo.
JUDGE_MODEL = "mistral-large-latest"

judge_llm = LangchainLLMWrapper(
    ChatMistralAI(model=JUDGE_MODEL, api_key=settings.MISTRAL_API_KEY, temperature=0)
)
judge_embeddings = LangchainEmbeddingsWrapper(ProjectEmbeddings())
print(f"Juge : {JUDGE_MODEL}")

Juge : mistral-large-latest


C:\Users\aamllal\AppData\Local\Temp\ipykernel_18796\2393859648.py:24: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(ProjectEmbeddings())


## 5. Évaluation RAGAS

In [ ]:
from ragas import EvaluationDataset, evaluate
from ragas.run_config import RunConfig
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

dataset = EvaluationDataset.from_list(samples)

metrics = [
    Faithfulness(),
    ResponseRelevancy(),
    LLMContextPrecisionWithReference(),
    LLMContextRecall(),
]

# max_workers bas : l'API Mistral (free/dev tier) limite le débit ; évite les 429.
# raise_exceptions=True : par défaut ragas avale les erreurs par métrique/échantillon et
# renvoie nan — on l'active temporairement pour voir la vraie cause si tout est nan.
result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
    run_config=RunConfig(max_workers=4),
    raise_exceptions=True,
)
result

## 6. Résultats détaillés et sauvegarde

Scores par question (1.0 = parfait) + moyennes. Le CSV horodaté permet de comparer les runs
(avant/après un changement de `RAG_TOP_K`, de modèle d'embeddings, de prompt, etc.).

In [ ]:
import pandas as pd

df = result.to_pandas()
metric_cols = [c for c in df.columns if c in {
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
}]

print("=== Scores moyens ===")
print(df[metric_cols].mean().round(3).to_string())

with pd.option_context("display.max_colwidth", 80):
    display(df[["user_input", *metric_cols]].round(3))

In [ ]:
from datetime import datetime

out_dir = BACKEND.parent / "notebooks" / "benchmark_results"
out_dir.mkdir(exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = out_dir / f"ragas_{stamp}.csv"
df.to_csv(out_path, index=False, encoding="utf-8")
print(f"Résultats enregistrés : {out_path}")

## 7. Lecture des résultats

- **Faithfulness bas, recall haut** : le retrieval ramène la bonne info mais le LLM hallucine
  ou extrapole → durcir le `SYSTEM_PROMPT`, baisser la température.
- **Context recall bas** : le retrieval rate les bons passages → jouer sur `RAG_TOP_K`, le
  chunking, ou la pondération dense/BM25 (RRF).
- **Context precision bas** : trop de passages hors-sujet remontés → baisser `top_k` ou
  ajouter un re-ranking.
- **Response relevancy bas** : la réponse part à côté de la question même avec un bon contexte
  → prompt de génération.

**Limites** : (1) le juge est lui-même un LLM (Mistral) — les scores sont indicatifs, pas
absolus ; (2) ces métriques portent sur le **texte** ; l'évaluation de la pertinence des
**figures** retournées nécessite un protocole dédié (hors périmètre RAGAS standard) ; (3)
étoffer le jeu d'évaluation est le levier n° 1 pour des chiffres fiables.